# Payment Fraud Detection — Exploratory Data Analysis

Run `python run_pipeline.py` first for full artifacts, or execute these cells on raw/clean data.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.ingest import load_transactions
from src.data.validate import run_validation
from src.features.engineering import engineer_features, FEATURE_COLUMNS
from src.utils.paths import CLEAN_CSV

sns.set_theme(style="whitegrid")

In [ ]:
if CLEAN_CSV.exists():
    df = pd.read_csv(CLEAN_CSV, parse_dates=["timestamp"])
else:
    raw = load_transactions()
    df, _, _ = run_validation(raw)

print(f"Rows: {len(df):,} | Fraud rate: {df['is_fraud'].mean():.4%}")
df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
df["is_fraud"].value_counts().sort_index().plot(kind="bar", ax=ax, color=["#2A9D8F", "#E76F51"])
ax.set_xticklabels(["Legit", "Fraud"], rotation=0)
ax.set_title("Class Balance")
plt.show()

print("Amount stats by class:")
df.groupby("is_fraud")["amount"].describe()

In [ ]:
for col in ["location", "payment_method", "device_type"]:
    rates = df.groupby(col)["is_fraud"].mean().sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(8, 3))
    rates.plot(kind="bar", ax=ax, color="#264653")
    ax.set_ylabel("Fraud Rate")
    ax.set_title(f"Fraud Rate by {col}")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
tmp = df.copy()
tmp["hour"] = tmp["timestamp"].dt.hour
hourly = tmp.groupby("hour")["is_fraud"].mean()
fig, ax = plt.subplots(figsize=(7, 3))
hourly.plot(marker="o", ax=ax, color="#E76F51")
ax.set_title("Fraud Rate by Hour of Day")
ax.set_xlabel("Hour")
ax.set_ylabel("Fraud Rate")
plt.show()

In [ ]:
feat = engineer_features(df.head(3000))
print(f"Engineered {len(FEATURE_COLUMNS)} model features")
feat[FEATURE_COLUMNS].describe().T